# E3-E2 — Baselines tabulaires multidevices (105 entrées)

**Objectif :** établir la comparaison C2 à **information disponible identique** face aux GCN relationnels : les mêmes 7 appareils × 5 instants × 3 variables, et les **sept températures futures** en sortie.

- Modèles : Ridge, Random Forest et XGBoost **multi-sorties**.
- Horizons : 15, 30, 60, 120 min.
- Entraînement : snapshots complets propres à chaque horizon.
- Sélection : RMSE de validation **sur les sept sorties**, sans utilisation du test.
- Évaluation : **13 643 observations nodales communes** (1 949 snapshots complets si toutes les clés du test sont présentes).
- Les résultats historiques sont conservés.
- **Attention :** cette expérience contrôle l'accès à l'information, mais pas le nombre de paramètres ni le biais inductif. Les conclusions ne doivent pas être attribuées uniquement à la représentation sémantique.

La matrice graphique est réellement stockée en `[snapshots, 5, 7, 3]` (et non `[snapshots, 7, 5, 3]`).


In [ ]:
from pathlib import Path
import json, time, platform
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error
from sklearn.base import clone
from sklearn import __version__ as sklearn_version
try:
    from xgboost import XGBRegressor
    import xgboost
except ImportError as exc:
    raise ImportError('Install xgboost in this Jupyter environment.') from exc

ROOT=Path.cwd().resolve()
REP=ROOT/'outputs/E3_aligned_representations'
COMMON=ROOT/'outputs/E3_common_evaluation'
OUT=ROOT/'outputs/E3_E2_multidevice_tabular_105'
HORIZONS=[15,30,60,120]
SPLITS=['train','validation','test']
SEED=42
EXPECTED_COMMON={'train':64393,'validation':13727,'test':13643}
assert REP.is_dir() and COMMON.is_dir()
OUT.mkdir(parents=True,exist_ok=True)
common=pd.read_csv(COMMON/'01_common_keys_all_horizons.csv')
common['timestamp']=pd.to_datetime(common.timestamp,utc=True)
assert len(common)==sum(EXPECTED_COMMON.values())
assert not common.duplicated(['uuid','timestamp','split']).any()
print('ROOT:',ROOT,'XGBoost:',xgboost.__version__)


In [ ]:
def read_horizon(h):
    folder=REP/f'h{h:03d}'
    node_df=pd.read_csv(folder/'01_graph_node_index.csv').sort_values('node_index')
    devices=node_df.uuid.astype(str).tolist()
    assert len(devices)==7 and len(set(devices))==7
    data={}
    for split in SPLITS:
        Xg=np.load(folder/f'X_graph_{split}_scaled.npy',mmap_mode='r')
        Xraw=np.load(folder/f'X_graph_{split}_unscaled.npy',mmap_mode='r')
        yg=np.load(folder/f'y_graph_{split}.npy',mmap_mode='r')
        meta=pd.read_csv(folder/f'metadata_{split}.csv')
        meta['timestamp']=pd.to_datetime(meta.timestamp,utc=True)
        meta['target_timestamp']=pd.to_datetime(meta.target_timestamp,utc=True)
        indices=np.load(COMMON/f'indices_h{h:03d}_{split}.npy')
        S=len(yg);N=7*S
        assert Xg.shape==Xraw.shape==(S,5,7,3),f'Unexpected graph tensor {Xg.shape}'
        assert yg.shape==(S,7) and len(meta)==N
        assert len(indices)==EXPECTED_COMMON[split]
        assert meta.uuid.astype(str).to_numpy().reshape(S,7).tolist()==[devices]*S
        assert (meta.target_timestamp-meta.timestamp).eq(pd.Timedelta(minutes=h)).all()
        assert np.allclose(meta.target.to_numpy(),yg.reshape(-1),rtol=0,atol=1e-5)
        got=meta.iloc[indices][['uuid','timestamp','split']].sort_values(['uuid','timestamp','split']).reset_index(drop=True)
        expected=common.loc[common.split.eq(split),['uuid','timestamp','split']].sort_values(['uuid','timestamp','split']).reset_index(drop=True)
        assert got.equals(expected)
        assert np.isfinite(Xg).all() and np.isfinite(yg).all()
        # Graph order [snapshot, time, node, variable] -> flatten 105 in this exact order.
        X=np.asarray(Xg).reshape(S,105).astype(np.float32)
        Y=np.asarray(yg).astype(np.float32)
        persistence=np.asarray(Xraw[:,-1,:,0],dtype=np.float64)
        data[split]=dict(X=X,Y=Y,meta=meta,indices=indices,persistence=persistence)
        print(f'h={h}, {split}: {S} snapshots; {N} nodes; {len(indices)} common nodes')
    return data,devices

def metrics(y,p):
    y=np.asarray(y,dtype=np.float64).reshape(-1)
    p=np.asarray(p,dtype=np.float64).reshape(-1)
    e=p-y;den=np.sum((y-y.mean())**2)
    return {'n':len(y),'RMSE':float(np.sqrt(np.mean(e**2))),
            'MAE':float(np.mean(np.abs(e))),
            'R2':float(1-np.sum(e**2)/den) if den>0 else np.nan,
            'Bias':float(e.mean())}

# Small, documented, explicit search space. This is a NEW controlled C2 experiment,
# not a claim of exact hyperparameter equivalence to historical 08A.
GRID={
 'Ridge':[
   {'alpha':0.01},{'alpha':0.1},{'alpha':1.0},{'alpha':10.0},{'alpha':100.0}
 ],
 'RandomForest':[
   {'n_estimators':150,'max_depth':12,'min_samples_leaf':2},
   {'n_estimators':150,'max_depth':20,'min_samples_leaf':2},
   {'n_estimators':250,'max_depth':12,'min_samples_leaf':1},
   {'n_estimators':250,'max_depth':20,'min_samples_leaf':1}
 ],
 'XGBoost':[
   {'n_estimators':250,'max_depth':3,'learning_rate':0.05},
   {'n_estimators':400,'max_depth':3,'learning_rate':0.05},
   {'n_estimators':250,'max_depth':5,'learning_rate':0.05},
   {'n_estimators':400,'max_depth':5,'learning_rate':0.05}
 ]
}

def make_model(family,params):
    if family=='Ridge':
        return Ridge(**params)
    if family=='RandomForest':
        return RandomForestRegressor(**params,random_state=SEED,n_jobs=-1)
    if family=='XGBoost':
        # Independent regressors, one per node; all receive the SAME 105 features.
        base=XGBRegressor(**params,objective='reg:squarederror',
                          random_state=SEED,n_jobs=2,tree_method='hist')
        return MultiOutputRegressor(base,n_jobs=2)
    raise ValueError(family)


## Entraînement

Les modèles sont entraînés sur les **snapshots** complets. L'évaluation principale s'effectue sur les **indices nodaux communs** pour garantir le même ensemble de cibles que dans E3-A/B/C/D. Chaque modèle prédit simultanément les sept températures.

**Coût :** XGBoost entraîne sept régressions par configuration ; l'exécution peut prendre plusieurs minutes. Aucune métrique de test n'est utilisée pour sélectionner les hyperparamètres.


In [ ]:
search_rows=[];metric_rows=[];pred_frames=[];device_rows=[];chosen_rows=[]
for h in HORIZONS:
    print(f'\\n===== HORIZON {h} MIN =====')
    data,devices=read_horizon(h)
    train,val,test=(data[s] for s in SPLITS)
    for family,configs in GRID.items():
        candidates=[]
        for i,params in enumerate(configs,1):
            model=make_model(family,params)
            t0=time.perf_counter()
            model.fit(train['X'],train['Y'])
            vp=np.asarray(model.predict(val['X']))
            assert vp.shape==val['Y'].shape
            vm=metrics(val['Y'],vp)
            row={'horizon_min':h,'model':family,'config_id':i,
                 'params':json.dumps(params,sort_keys=True),
                 'validation_RMSE':vm['RMSE'],'validation_MAE':vm['MAE'],
                 'fit_seconds':time.perf_counter()-t0}
            search_rows.append(row);candidates.append(row)
            print(f'{family} config {i}: val RMSE={vm["RMSE"]:.6f}')
            del model
        best=min(candidates,key=lambda r:(r['validation_RMSE'],r['config_id']))
        params=json.loads(best['params'])
        model=make_model(family,params)
        start=time.perf_counter()
        model.fit(train['X'],train['Y']) # no train+validation refit
        elapsed=time.perf_counter()-start
        chosen_rows.append({'horizon_min':h,'model':family,'config_id':best['config_id'],
                            'params':best['params'],'selection_validation_RMSE':best['validation_RMSE'],
                            'fit_seconds':elapsed,'train_snapshots':len(train['Y']),
                            'validation_snapshots':len(val['Y']),
                            'test_snapshots':len(test['Y'])})
        for split in SPLITS:
            d=data[split]
            pred=np.asarray(model.predict(d['X']),dtype=np.float64)
            assert pred.shape==d['Y'].shape and np.isfinite(pred).all()
            y=d['Y'].reshape(-1);p=pred.reshape(-1)
            ref=d['persistence'].reshape(-1)
            for scope,ix in [('full_horizon_split',np.arange(len(y))),
                             ('common_keys',d['indices'])]:
                if scope=='common_keys' and split!='test':continue
                mm=metrics(y[ix],p[ix]);rr=metrics(y[ix],ref[ix])
                metric_rows.append({'horizon_min':h,'model':family,'split':split,
                                    'scope':scope,**mm,
                                    'Persistence_RMSE':rr['RMSE'],
                                    'skill_RMSE':1-mm['RMSE']/rr['RMSE']})
            if split=='test':
                ix=d['indices']
                df=d['meta'].iloc[ix].copy().reset_index(drop=True)
                df['horizon_min']=h;df['model']=family
                df['y_true']=y[ix];df['y_pred']=p[ix]
                df['persistence_pred']=ref[ix];df['error']=p[ix]-y[ix]
                pred_frames.append(df)
                for uuid,g in df.groupby('uuid',sort=True):
                    mm=metrics(g.y_true,g.y_pred)
                    rr=metrics(g.y_true,g.persistence_pred)
                    device_rows.append({'horizon_min':h,'model':family,'uuid':uuid,
                                        **mm,'Persistence_RMSE':rr['RMSE'],
                                        'skill_RMSE':1-mm['RMSE']/rr['RMSE']})
        print('FINAL',h,family,'selected',best['config_id'],'validation',best['validation_RMSE'])
        del model

search=pd.DataFrame(search_rows)
all_metrics=pd.DataFrame(metric_rows)
predictions=pd.concat(pred_frames,ignore_index=True)
devices_df=pd.DataFrame(device_rows)
chosen=pd.DataFrame(chosen_rows)
primary=all_metrics.query("split=='test' and scope=='common_keys'").copy()
assert len(primary)==12 and primary.n.eq(13643).all()
assert len(predictions)==4*3*13643
assert not predictions.duplicated(['horizon_min','model','uuid','timestamp']).any()
assert len(devices_df)==4*3*7
search.to_csv(OUT/'02_validation_search.csv',index=False)
all_metrics.to_csv(OUT/'01_metrics_all_splits.csv',index=False)
predictions.to_csv(OUT/'03_common_test_predictions.csv',index=False)
devices_df.to_csv(OUT/'04_common_test_per_device.csv',index=False)
chosen.to_csv(OUT/'05_selected_configs_and_training.csv',index=False)
primary.to_csv(OUT/'00_PRIMARY_common_TEST_results.csv',index=False)
protocol={
 'experiment':'E3-E2, C2 controlled information, non-graph multi-device baselines',
 'horizons_minutes':HORIZONS,
 'input':'X_graph_scaled [snapshots,5,7,3] flattened to [snapshots,105]',
 'output':'seven future device temperatures [snapshots,7]',
 'node_order':'01_graph_node_index.csv, node_index ascending',
 'training':'full horizon-specific train snapshots',
 'selection':'validation RMSE across all 7 outputs, no test selection',
 'evaluation':'common test indices: 13643 node-level samples per horizon/model',
 'random_seed':SEED,
 'grid':GRID,
 'modeling_note':'XGBoost is seven independent regressors sharing the same 105 inputs',
 'comparison_note':'information controlled; architecture and parameter count not controlled',
 'historical_results':'unchanged',
 'python':platform.python_version(),
 'sklearn':sklearn_version,
 'xgboost':xgboost.__version__
}
(OUT/'06_protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
print(primary[['horizon_min','model','RMSE','MAE','R2','Bias','skill_RMSE']].to_string(index=False))
print('Saved:',OUT)


## Fichiers à transmettre

- `00_PRIMARY_common_TEST_results.csv`
- `02_validation_search.csv`
- `04_common_test_per_device.csv`
- `05_selected_configs_and_training.csv`
- `06_protocol.json`

Après E3-E2, entraîner les variantes LSTM et TCN multidevices sur les **mêmes 105 valeurs**, puis rapprocher ces résultats des GCN. La comparaison C2 contrôle les entrées disponibles, mais il reste à auditer les modèles GCN déjà entraînés, leur convergence et l'identité exacte des échantillons de test.
